# Exercise 2.

In [1]:
# --- Read the BLOSUM62 matrix from .txt file ---

# Read and store the whole blosum62.txt file as a list of strings, each string representing a row

f = open("blosum62.txt", "r")
full_file_blosum62 = f.readlines()
f.close()

In [2]:
# --- Create score matrix dictionary ---

# --- Separate the residues ---

# Only looking at second line containing the residues
# Filter out the white space and newline at the end, store each letter as individual string

residues = full_file_blosum62[1].split()


# --- Store the matrix values as nested list (list of lists) --- 

# Each row of values in the matrix is stored as a list of value

score_matrix = []

for row_str in full_file_blosum62[2:]:
    
    # Turn row of string into list of string elements
    row_list = row_str.split()

    row_value_list = []

    for elem in row_list[1:]:
        # Skipping first element which is a residue letter

        row_value_list.append(int(elem))

    score_matrix.append(row_value_list)


# --- Turn into dictionary ---

# Since the list of residues have indices corresponding to the position in the scoring matrix we can easily construct a dictionary with
# pairs of residues (as tuple of two string) and their score

blosum62_score_matrix_dict = {}

for i, resA in enumerate(residues):
    for j, resB in enumerate(residues):
        blosum62_score_matrix_dict[(resA, resB)] = score_matrix[i][j]

In [3]:
blosum62_score_matrix_dict.get(("W", "W"))

11

In [ ]:
# Functions

def faa_fasta_reader(full_filename):
    """ Return sequence as string of residues """

    f = open(full_filename, "r")
    full_file = f.readlines()
    f.close()

    seq = ""

    # Skip first line in file
    for row_string in full_file[1:]:
        for res in row_string:
            if res == " " or res == "\n":
                pass

            else:
                seq += res

    return seq

def initial_matrix_setup(seqA, seqB, gap_open_pen, gap_extend_pen):

    if gap_open_pen > 0 or gap_extend_pen > 0:
        raise ValueError("Penalties must strictly be negative")

    h = gap_open_pen
    s = gap_extend_pen

    n_rows = len(seqB) + 2
    n_columns = len(seqA) + 2

    V_matrix = []
    E_matrix = []
    F_matrix = []
    G_matrix = []
    backtrack_matrix = []

    for i in range(n_rows):
        row_list_V = []
        row_list_E = []
        row_list_F = []
        row_list_G = []
        row_list_backtrack = []

        for j in range(n_columns):
            row_list_V.append(0)
            row_list_E.append(0)
            row_list_F.append(0)
            row_list_G.append(0)
            row_list_backtrack.append(0)

        V_matrix.append(row_list_V)
        E_matrix.append(row_list_E)
        F_matrix.append(row_list_F)
        G_matrix.append(row_list_G)
        backtrack_matrix.append(row_list_backtrack)

    V_matrix[0][0] = E_matrix[0][0] = F_matrix[0][0] = G_matrix[0][0] = " "
    V_matrix[0][1] = E_matrix[0][1] = F_matrix[0][1] = G_matrix[0][1] = "-"
    V_matrix[1][0] = E_matrix[1][0] = F_matrix[1][0] = G_matrix[1][0] = "-"
    V_matrix[1][1] = E_matrix[1][1] = F_matrix[1][1] = G_matrix[1][1] = 0

    # Filling in row seq. in first row
    for i in range(len(seqA)):
        V_matrix[0][i+2] = seqA[i]
        E_matrix[0][i+2] = seqA[i]
        F_matrix[0][i+2] = seqA[i]
        G_matrix[0][i+2] = seqA[i]

    # Filling in column seq. in first column
    for j in range(len(seqB)):
        V_matrix[j+2][0] = seqB[j]
        E_matrix[j+2][0] = seqB[j]
        F_matrix[j+2][0] = seqB[j]
        G_matrix[j+2][0] = seqB[j]

    # V matrix
    for i in range(2, n_rows):
        # V_matrix[i][1] = h + (i-1)*s
        V_matrix[i][1] = 0

    for j in range(2, n_columns):
        # V_matrix[1][j] = h + (j-1)*s
        V_matrix[1][j] = 0

    # E matrix
    for i in range(2, n_rows):
        # E_matrix[i][1] = 2*h + i*s
        E_matrix[i][1] = 0

    for j in range(2, n_columns):
        E_matrix[1][j] = -99
        # E_matrix[1][j] = 0

    # F matrix
    for i in range(2, n_rows):
        F_matrix[i][1] = -99
        # F_matrix[i][1] = 0

    for j in range(2, n_columns):
        # F_matrix[1][j] = 2*h + j*s
        F_matrix[1][j] = 0
    
    # G matrix
    for i in range(2, n_rows):
        # G_matrix[i][1] = h + (i-1)*s
        G_matrix[i][1] = 0

    for j in range(2, n_columns):
        # G_matrix[1][j] = h + (j-1)*s
        G_matrix[1][j] = 0

    return V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix

def local_alignment_algo(V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix, seqA, seqB, h, s):
    # Element format (list): [value, i, j]
    highest_value_index = [[0, 0, 0]]

    for i in range(2, len(seqB)+2):
        for j in range(2, len(seqA)+2):
            
            G_val = V_matrix[i-1][j-1] + blosum62_score_matrix_dict.get((V_matrix[i][0], V_matrix[0][j]))

            G_matrix[i][j] = G_val

            # Max value E
            E_val_ext = E_matrix[i][j-1] + s
            E_val_open = V_matrix[i][j-1] + h + s

            if E_val_ext > E_val_open:
                E_val = E_val_ext
                E_val_symb = "E_ext"
            else:
                E_val = E_val_open
                E_val_symb = "E_open"

            E_matrix[i][j] = E_val

            # Max value F
            F_val_ext = F_matrix[i-1][j] + s
            F_val_open = V_matrix[i-1][j] + h + s

            if F_val_ext > F_val_open:
                F_val = F_val_ext
                F_val_symb = "F_ext"
            else:
                F_val = F_val_open
                F_val_symb = "F_open"

            F_matrix[i][j] = F_val

            # --- Prioritizing logic for backtrack matrix symbol ---
            
            # In case best value is lower than zero, bump it up to zero as essential in the Smith-Waterman algorithm
            # This would be an endpoint, so no need to record direction
            if G_val < 0 and E_val < 0 and F_val < 0:
                V_matrix[i][j] = 0
                backtrack_matrix[i][j] = "Stop"

            else:

                # Must prioritize "E/F_open" if E_val or F_val are equal or larger than G_val, so gaps can be close in backtrack

                if G_val > E_val and G_val > F_val:
                    V_matrix[i][j] = G_val
                    backtrack_matrix[i][j] = "G"

                # If same value in G and E (both grater than F) and we have a "E_open" we prioritize it so gaps can be closed in during backtrack
                elif G_val == E_val and G_val > F_val:
                    if E_val_symb == "E_open":
                        V_matrix[i][j] = E_val
                        backtrack_matrix[i][j] = "E_open"
                    # If not we prefer match/missmatch
                    else:
                        V_matrix[i][j] = G_val
                        backtrack_matrix[i][j] = "G"
                # Same for F regarding F_open
                elif G_val == F_val and G_val > E_val:
                    if F_val_symb == "F_open":
                        V_matrix[i][j] = F_val
                        backtrack_matrix[i][j] = "F_open"
                    # If not we prefer match/missmatch
                    else:
                        V_matrix[i][j] = G_val
                        backtrack_matrix[i][j] = "G"
                # In case all ties, we look if there is an "_open" tag, if not prioritize G_val i.e. match/missmatch 
                elif G_val == E_val and G_val == F_val:
                    if E_val_symb == "E_open" and F_val_symb == "F_open":
                        # Prioritize gap in shortest sequence
                        if len(seqA) > len(seqB):
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb
                        elif len(seqB) > len(seqA):
                            V_matrix[i][j] = F_val
                            backtrack_matrix[i][j] = F_val_symb
                        # If both seqs. are same length, prefer gap in column seq.
                        elif len(seqB) == len(seqA):
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb

                    elif E_val_symb == "E_open" or F_val_symb == "F_open":
                        if E_val_symb == "E_open":
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb
                        elif F_val_symb == "F_open":
                            V_matrix[i][j] = F_val
                            backtrack_matrix[i][j] = F_val_symb

                    else:    
                        V_matrix[i][j] = G_val
                        backtrack_matrix[i][j] = "G"
                    
                elif E_val > G_val and E_val > F_val:
                    V_matrix[i][j] = E_val
                    backtrack_matrix[i][j] = E_val_symb

                elif F_val > G_val and F_val > E_val:
                    V_matrix[i][j] = F_val
                    backtrack_matrix[i][j] = F_val_symb

                # If ties between gaps, prefer gap in shortest seq.                
                elif E_val == F_val and E_val > G_val:
                    
                    if E_val_symb == "E_open" and F_val_symb == "F_open":
                        if len(seqA) > len(seqB):
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb
                        elif len(seqB) > len(seqA):
                            V_matrix[i][j] = F_val
                            backtrack_matrix[i][j] = F_val_symb
                        # If seqs. are same len, prefer column seq. 
                        elif len(seqB) == len(seqA):
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb

                    elif E_val_symb == "E_open" or F_val_symb == "F_open":
                        if E_val_symb == "E_open":
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb
                        elif F_val_symb == "F_open":
                            V_matrix[i][j] = F_val
                            backtrack_matrix[i][j] = F_val_symb
                    #If we have "E/F_ext"
                    else:
                        if len(seqA) > len(seqB):
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb
                        elif len(seqB) > len(seqA):
                            V_matrix[i][j] = F_val
                            backtrack_matrix[i][j] = F_val_symb
                        # If seqs. are same len, prefer column seq. 
                        elif len(seqB) == len(seqA):
                            V_matrix[i][j] = E_val
                            backtrack_matrix[i][j] = E_val_symb

                else:
                    print("Some configuration not picked and counted for!")
                    print(f"G_val: {G_val}")
                    print(f"E_val: {E_val}, E_val_symb: {E_val_symb}")
                    print(f"F_val: {F_val}, F_val_symb: {F_val_symb}")
                    break

            # Check against the latest added highest element: add to list if equal, overwrite all if higher
            if V_matrix[i][j] > highest_value_index[-1][0]:
                highest_value_index = [[V_matrix[i][j], i, j]]

            elif V_matrix[i][j] == highest_value_index[-1][0]:
                highest_value_index.append([V_matrix[i][j], i, j])

    print("Assigning scoring completed!")

    return V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix, highest_value_index

def backtrack_algo(V_matrix, backtrack_matrix, highest_value_index, seqA, seqB):
    alignment_seqA = ""
    alignment = ""
    alignment_seqB = ""

    i = highest_value_index[0][1]
    j = highest_value_index[0][2]

    E_gap = "off"   # on/off
    F_gap = "off"   # on/off

    first_backtrack_direction = backtrack_matrix[i][j]
    if first_backtrack_direction == "E_ext":
        E_gap = "on"

    if first_backtrack_direction == "F_ext":
        F_gap = "on"

    while i > 1  or j > 1:

        if E_gap == "on" and F_gap == "on":
            print("Both gaps on", i, j)
            break

        if V_matrix[i][j] == 0 or backtrack_matrix[i][j] == "Stop":
            print(f"Stopped at i: {i}, j: {j}")
            print(f"V_matrix: {V_matrix[i][j]}")
            print(f"backtrack_matrix: {backtrack_matrix[i][j]}")
            break

        backtrack_direction = backtrack_matrix[i][j]

        # Check if we are in a gap or not
        if E_gap == "on":
            alignment_seqA += seqA[j-2]
            alignment += " "
            alignment_seqB += "-"
            j -= 1

            # If we were in a gap in E and we have E_open we know the gap started here, can turn it off
            if backtrack_direction == "E_open":
                E_gap = "off"
            else:
                # E_gap stays on
                pass
        
        elif F_gap == "on":
            alignment_seqB += seqB[i-2]
            alignment += " "
            alignment_seqA += "-"
            i -= 1

            if backtrack_direction == "F_open":
                F_gap = "off"
            else:
                # F_gap stays on
                pass
        
        # If not in a gap
        else:

            if backtrack_direction == "E_open":
                alignment_seqA += seqA[j-2]
                alignment += " "
                alignment_seqB += "-"
                j -= 1
                E_gap = "off"

            elif backtrack_direction == "E_ext":
                alignment_seqA += seqA[j-2]
                alignment += " "
                alignment_seqB += "-"
                j -= 1
                E_gap = "on"

            elif backtrack_direction == "F_open":
                alignment_seqB += seqB[i-2]
                alignment += " "
                alignment_seqA += "-"
                i -= 1
                F_gap = "off"
            
            elif backtrack_direction == "F_ext":
                alignment_seqB += seqB[i-2]
                alignment += " "
                alignment_seqA += "-"
                i -= 1
                F_gap = "on"
            
            elif backtrack_direction == "G":
                alignment_seqA += seqA[j-2]
                alignment_seqB += seqB[i-2]

                if (seqA[j-2] == seqB[i-2]):
                    match_residue = seqA[j-2]
                    alignment += f"{match_residue}"
                else:
                    alignment += " "

                i -= 1
                j -= 1

                E_gap = "off"
                F_gap = "off"

    corrected_alignment_seqA = alignment_seqA[::-1]
    corrected_alignment_symb = alignment[::-1]
    corrected_alignment_seqB = alignment_seqB[::-1]

    return corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB

def print_alignment(corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB):
    n_alignments = len(corrected_alignment_symb)
    start_of_int = 0
    for i in range(n_alignments):
        if i > 0 and i%59 == 0:
            print(corrected_alignment_seqA[start_of_int:i+1])
            print(corrected_alignment_symb[start_of_int:i+1])
            print(corrected_alignment_seqB[start_of_int:i+1])

            print("\n")

            start_of_int = i

        elif i == n_alignments-1:
            print(corrected_alignment_seqA[start_of_int:])
            print(corrected_alignment_symb[start_of_int:])
            print(corrected_alignment_seqB[start_of_int:])

def calculate_alignment_score(corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB):
    sum = 0
    gap_openings = 0
    for i, symb in enumerate(corrected_alignment_symb):
        if symb == " ":

            if corrected_alignment_seqA[i] == "-":
                if corrected_alignment_seqA[i-1] == "-":
                    sum += s
                else:
                    sum += h + s
                    gap_openings += 1

            elif corrected_alignment_seqB[i] == "-":
                if corrected_alignment_seqB[i-1] == "-":
                    sum += s
                else:
                    sum += h + s
                    gap_openings += 1
            
            else:
                sum += blosum62_score_matrix_dict.get((corrected_alignment_seqA[i], corrected_alignment_seqB[i]))

        else:
            sum += blosum62_score_matrix_dict.get((symb, symb))

    return sum, gap_openings

## Exercise 2.

### b)

In [18]:
seqA = faa_fasta_reader("NP_081220_1.faa")
seqB = faa_fasta_reader("sequence_Q9SIE0_2.fasta")

h = -11 # opening pen
s = -1 # extend pens

V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix = initial_matrix_setup(seqA, seqB, gap_open_pen=h, gap_extend_pen=s)

V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix, highest_value_index = local_alignment_algo(V_matrix, 
                                                                                                    E_matrix,
                                                                                                    F_matrix,
                                                                                                    G_matrix,
                                                                                                    backtrack_matrix,
                                                                                                    seqA,
                                                                                                    seqB,
                                                                                                    h,
                                                                                                    s)

print(f"highest_value_index: {highest_value_index}")

corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB = backtrack_algo(V_matrix, 
                                                                                              backtrack_matrix, 
                                                                                              highest_value_index, 
                                                                                              seqA, 
                                                                                              seqB)


Assigning scoring completed!
highest_value_index: [[228, 314, 278]]
Stopped at i: 113, j: 96
V_matrix: 0
backtrack_matrix: Stop


In [16]:
print_alignment(corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB)

FVDLKEADWILEQLCKDVPWKQRMGIREDVTYPQPRLTAWYGE-----LPYTYSRITMEP
F            L K  PW             QPR T          L Y   R T   
FLPFQQSWTFFDYLDKHIPWTRPTIRVFGRSCLQPRDTCYVASSGLTALVYSGYRPTSYS


PNPHWLPVLWTLKSRIEENTSHTFNSLLCNFYRDEKDSVDWHSDDEPSLGSCPVIASLSF
      P    L           FNSLL N Y    D V WH DDE   G  P IAS SF
SWDDFPPLKEILDAIYKVLPGSRFNSLLLNRYKGASDYVAWHADDEKIYGPTPEIASVSF


FGATRTFEMRKKPPPE----ENGD----------YTYVERVKIPLDHGTLLIMEGATQAD
FG  R F   KK   E      GD                    L HG LL M G TQ D
FGCERDFVLKKKKDEESSQGKTGDSGPAKKRLKRSSREDQQSLTLKHGSLLVMRGYTQRD


DWQHRVPKEYHSRQPRVNLTFRTV
DW H VPK       R NLTFR V
DWIHSVPKRAKAEGTRINLTFRLV


In [21]:
sum, gap_openings = calculate_alignment_score(corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB)
print(sum)
print(gap_openings)

228
3


### c)

In [22]:
seqA = faa_fasta_reader("NP_081220_1.faa")
seqB = faa_fasta_reader("P05050.faa")

h = -11 # opening pen
s = -1 # extend pens

V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix = initial_matrix_setup(seqA, seqB, gap_open_pen=h, gap_extend_pen=s)

V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix, highest_value_index = local_alignment_algo(V_matrix, 
                                                                                                    E_matrix,
                                                                                                    F_matrix,
                                                                                                    G_matrix,
                                                                                                    backtrack_matrix,
                                                                                                    seqA,
                                                                                                    seqB,
                                                                                                    h,
                                                                                                    s)

print(f"highest_value_index: {highest_value_index}")

corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB = backtrack_algo(V_matrix, 
                                                                                              backtrack_matrix, 
                                                                                              highest_value_index, 
                                                                                              seqA, 
                                                                                              seqB)


Assigning scoring completed!
highest_value_index: [[48, 211, 276]]
Stopped at i: 131, j: 191
V_matrix: 0
backtrack_matrix: Stop


In [23]:
print_alignment(corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB)

HSD-DEPSLGSCPVIASLSFGATRTFEMRKKPPPEENGDYTYVERVKIPLDHGTLLIMEG
H D DEP L    V  SL   A   F   K   P               L HG      G
HQDKDEPDLRAPIVSVSLGLPAIFQFGGLKRNDPLK----------RLLLEHGDVVVWGG


GATQADWQHRVPKEYHSRQP------------RVNLTFR
G             YH  QP            R NLTFR
GESRL--------FYHGIQPLKAGFHPLTIDCRYNLTFR


In [24]:
sum, gap_openings = calculate_alignment_score(corrected_alignment_seqA, corrected_alignment_symb, corrected_alignment_seqB)
print(sum)
print(gap_openings)

48
4
